# 3. Keras

High-level TensorFlow API named Keras, focusing on the following:

* Keras Sequential API
* Keras Functional API
* Keras Subclassing API
* Keras Preprocessing API

# Introduction

Keras is an easy-to-use and accessible library to enable fast machine learning experiments. It's a high-level nerual network API with multiple backends. Keras can support multiple use cases including research, application development and model training

Benefits: 
* Modular
* Scalable
* Composable

TensorFlow Keras API is the same as the Keras API, but the TensorFlow version is optimized for the TF backend. It integrates TF-specific functionality. The true difference in building models between the 2 is just how it's imported

Keras:
```
import keras
```

TensorFlow Keras:
```
import tensorflow as tf
from tensorflow import keras
```

# Understanding Keras Layers

Layers are the fundamental building blocks of Keras models. They receive data as input, does a specific task and returns an output

Layers include:
* __Core Layers__: Dense, Activation, Flatten, Input, Reshape, Permute ...
* __Conv Layers__: Conv1D, Conv2D, SeparableConv1D, Conv3D, ...
* __Pooling__: Performs downsampling operation to reduce feature map. MaxPooling1D, AveragePooling2D, and GlobalAveragePooling3D
* __Reccurent Layers__: RNN, SimpleRNN, GRU, LSTM, ConvLSTM2D, ...
* __Embedding Layer__: Only used as the first later in a model. Turns positive integers into dense vectors of fixed size
* __Merge Layer__: Add, Subtract, Multiply, Average, Maximum, Minimum, ...
* __Advanced activation Layer__: LeakyReLU, PReLU, Softmax, ReLU, ...
* __Batch Normalization Layer__: Normalizes the activation of the previous layer at each batch
* __Noise Layer__: GausianNoise, GausianDropout, and AlphaDropout
* __Layer Wrappers__: TimeDistributed applies a layer to every temporal slice of an input and bidirectional wrapper for RNNs
* __Locally-connected Layers__: LocallyConnected1D and LocallyConnected2D. They work like Conv1D or Conv2D without sharing their weights

## Useful Layer Methods

* `layer.get_weights()` - returns the weights of the layer as a list of numpy arrays
* `layer.set_weights(weights)` - fixes the weights of the layer from a list of numpy arrays
* `layer.input` | `layer.output` - get the inputs and outputs of a layer
	- `layer.get_input_at(node_index) | layer.get_output_at(node_index) - if layer has multiple nodes
* `layer.input_shape` | `layer.output_shape` - layer's input and output shapes
	- `layer.get_input_shape_at(node_index) | layer.get_output_shape_at(node_index) - if layer has multiple nodes
* `layer.get_config()` - returns a dictionary containing the configuration of the layer. Does not include weights or connectivity information
* `layer.from_config(config)` - Instantiates layer's configuration. Configuration is stored in an associative array (Dict)

# 3a. Keras Sequential API

Create Sequential models - linear stack of layers to predict an output. Prediction via the compilation, training, and evaluation steps

In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense
import numpy as np

## Closer look at `tf.keras.layers` API

Contains a lof of built-in layers and API to configure the layers

1. Add __activation__ function by specifying the name of a built-in function or as a callable object. Determines if a neuron should be activiated or not. Default no activation function
2. __Initialization__ strategy by passing the string identifier of built-in initializaers or a callable object. Default is "Glorot uniform" initializer for weights, 0 for biases
3. __Regularizers__ for kernel and biases (L1, L2). Default no regularization

In [ ]:
# 1. Activation functions
Dense(256, activation='sigmoid')
Dense(256, activation=tf.keras.activations.sigmoid)

# 2. Initialization strategy
Dense(256, kernel_initializer="random_normal")
Dense(256, bias_initializer=tf.keras.initializers.Constant)

# 3. Regularizers
Dense(256, kernel_regularizer=tf.keras.regularizers.l1)
Dense(256, kernel_regularizer=tf.keras.regularizers.l2)

2 Ways to create a Sequential model:

1. Passing a list of layer instances as an array to the `Sequential` constructor
2. Instantiate a Sequential class then add the layers via `.add()` method

Recommended to specify the input dimensions on the first layer. If not specified, weights will not be generated and cannot use aggregation commands like `summary`, `layers`, `weights`. Usually input dimension will be batch size and number of features

In [5]:
# List of layers - multi-class classifier (10 categories) - Multi-layer perceptron
model = tf.keras.Sequential(
	[tf.keras.layers.Dense(1024, input_dim=64),
	tf.keras.layers.Activation('relu'),
	tf.keras.layers.Dense(256),
	tf.keras.layers.Activation('relu'),
	tf.keras.layers.Dense(10),
	tf.keras.layers.Activation('softmax')]
)

In [4]:
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(1024, input_dim=64))
# ...

## Compiling the Model with `model.compile`

Done using the `compile` method. Have to specify

* Optimization algorithm for training. Pass optimizer instance from `tf.keras.optimizers` module
* Loss function. Algorithm used for minimizing the loss function
* List of metrics to judge the model performance that aren't used in the training process
* Model trains and evaluates eagerly use `run_eagerly`

In [6]:
# Compiling the model with parameters
model.compile(
	optimizer='adam',
 	loss='categorical_crossentropy',
  	metrics=['accuracy']
)

In [8]:
# Generate 3 toy datasets with 64 features. Train, validate, test
data = np.random.random((2000, 64))
labels = np.random.random((2000, 10))
val_data = np.random.random((500, 64))
val_labels = np.random.random((500, 10))
test_data = np.random.random((500, 64))
test_labels = np.random.random((500, 10))

## Learning, Evaluating, and Testing

Once model is configured, it will be trained, then tuned using the validation set and finally tested and measures against the test set

In [10]:
# Model fitting/training. Validation run at the end of each epoch
model.fit(
	data,
	labels,
	epochs=10,
	batch_size=50,
	validation_data=(val_data, val_labels)
)

# Model evaluation
model.evaluate(data, labels, batch_size=50)

# Model prediction
result = model.predict(data, batch_size=50)
print(result)

Epoch 1/10
40/40 [==============================] - 0s 3ms/step - loss: 8990.1523 - accuracy: 0.0965 - val_loss: 7224.2222 - val_accuracy: 0.0880
Epoch 2/10
40/40 [==============================] - 0s 3ms/step - loss: 10284.6328 - accuracy: 0.0910 - val_loss: 10655.6514 - val_accuracy: 0.1080
Epoch 3/10
40/40 [==============================] - 0s 3ms/step - loss: 12651.4277 - accuracy: 0.0855 - val_loss: 16420.9062 - val_accuracy: 0.1080
Epoch 4/10
40/40 [==============================] - 0s 3ms/step - loss: 20425.0371 - accuracy: 0.0905 - val_loss: 21017.6523 - val_accuracy: 0.0860
Epoch 5/10
40/40 [==============================] - 0s 3ms/step - loss: 19092.4219 - accuracy: 0.0960 - val_loss: 27147.2031 - val_accuracy: 0.1080
Epoch 6/10
40/40 [==============================] - 0s 3ms/step - loss: 25102.0391 - accuracy: 0.0965 - val_loss: 36152.5469 - val_accuracy: 0.1080
Epoch 7/10
40/40 [==============================] - 0s 4ms/step - loss: 23279.2188 - accuracy: 0.1010 - val_loss: 

# 2b. Keras Functional API

Sequential API limited to a linear topology. Functional API provide non-linear topology (not just feed forward neural networks). Many high-performing models utilize non-linear topology. 

Deep learning model is usually a directed acyclic graph (DAG).

Below create a Functional model, using callable models, manipulating complex graph topologies, sharing layers and introducing the concepto of layer "node"

In [1]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input, Dense, TimeDistributed
import keras.models

## Creating a Functional model

MNIST dataset of handwritten digits as grayscale images to predict the number from human handwriting

In [16]:
# Load Mnist data
mnist = tf.keras.datasets.mnist
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = mnist.load_data()

# Create the input layer. Specify the input shape, should correspond to training data shape
inputs = tf.keras.Input(shape=(28,28))

# Flatten image to create array of 784 pixels
flatten_layer = keras.layers.Flatten()
flatten_output = flatten_layer(inputs)

# Dense layers
dense_layer = tf.keras.layers.Dense(50, activation="relu")
dense_output = dense_layer(flatten_output)

# Final layer - output
predictions = tf.keras.layers.Dense(10, activation='softmax')(dense_output)

model = keras.Model(inputs=inputs, outputs=predictions)
model.summary()

Model: "model_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input_4 (InputLayer)         [(None, 28, 28)]          0         
_________________________________________________________________
flatten_3 (Flatten)          (None, 784)               0         
_________________________________________________________________
dense_12 (Dense)             (None, 50)                39250     
_________________________________________________________________
dense_13 (Dense)             (None, 10)                510       
Total params: 39,760
Trainable params: 39,760
Non-trainable params: 0
_________________________________________________________________


In [17]:
# Compile and train model
model.compile(
	optimizer='sgd',
	loss='sparse_categorical_crossentropy',
	metrics=['accuracy']
)
model.fit(
	X_mnist_train,
	y_mnist_train,
	validation_data=(X_mnist_train, y_mnist_train),
	epochs=10
)

Epoch 1/10
1875/1875 [==============================] - 10s 5ms/step - loss: 30.0186 - accuracy: 0.1121 - val_loss: 2.3009 - val_accuracy: 0.1125
Epoch 2/10
1875/1875 [==============================] - 13s 7ms/step - loss: 2.2994 - accuracy: 0.1132 - val_loss: 2.2920 - val_accuracy: 0.1195
Epoch 3/10
1875/1875 [==============================] - 13s 7ms/step - loss: 2.2969 - accuracy: 0.1160 - val_loss: 2.2996 - val_accuracy: 0.1131
Epoch 4/10
1875/1875 [==============================] - 12s 7ms/step - loss: 2.2772 - accuracy: 0.1272 - val_loss: 2.2404 - val_accuracy: 0.1400
Epoch 5/10
1875/1875 [==============================] - 13s 7ms/step - loss: 2.2535 - accuracy: 0.1416 - val_loss: 2.2211 - val_accuracy: 0.1525
Epoch 6/10
1875/1875 [==============================] - 12s 7ms/step - loss: 2.2381 - accuracy: 0.1466 - val_loss: 2.2991 - val_accuracy: 0.1155
Epoch 7/10
1875/1875 [==============================] - 13s 7ms/step - loss: 2.3016 - accuracy: 0.1152 - val_loss: 2.2570 - val_a

## Using callable models like layers

1. Any model can be treated as a layer by calling it on a tensor. Model pretrained and assigned to a variable will produce the expected output and new layers can make use of the output for a new model. Model will reuse the architecture and trained weights
2. Easily apply wrappers over existing models to change the type of model. e.g adding a `TimeDistributed` layer wrapper to predict for videos instead of images


## Creating a model with multiple inputs and outputs

Easy to manipulate a large number of intertwined datastreams with multiple inputs and outputs in a non-linear connectivity topology.

Below build a system for predicting the price of a specific house and the elapsed time before its sale. 2 inputs & 2 outputs: 

__Inputs__
* Data about the house such as number of bedrooms, house size, ...
* Recent picture about the house

__Outputs__
* Elapsed time before the sale (slow / fast)
* Predicted price

In [2]:
import pydot
import graphviz

# First block to process the tabular data about the house
house_data_inputs = tf.keras.Input(shape=(128, ), name='house_data')
x = tf.keras.layers.Dense(64, activation='relu')(house_data_inputs)
block_1_output = tf.keras.layers.Dense(32, activation='relu')(x)

# Second block to process the house image data
house_picture_inputs = tf.keras.Input(shape=(128, 128, 3), name='house_picture')
x_pic = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(house_picture_inputs)
x_pic = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x_pic)
block_2_output = tf.keras.layers.Flatten()(x_pic)

# Merge all features into a single large vector by concatenation
x_merge = tf.keras.layers.concatenate([block_1_output, block_2_output])

# Logistic regression for price preduction 
price_pred = tf.keras.layers.Dense(1, name='price', activation='relu')(x_merge)
# Time classifier on top of features
time_elapsed_pred = tf.keras.layers.Dense(2, name='elapsed_time', activation='softmax')(x_merge)

# Build the model
model = keras.Model(
	[house_data_inputs, house_picture_inputs],
	[price_pred, time_elapsed_pred],
	name='toy_house_pred'
)
keras.utils.plot_model(model, 'multi_input_and_output_model.png', show_shapes=True)

InvocationException: GraphViz's executables not found

Functional models are able take in multiple inputs and process them separately, finally combining them for the final output(s)

## Shared Layers

Some models reuse the same layer multiple times inside the architecture. These layers learn features that correspond to multiple paths in the graph of layers. Shared layers are usually used to encode inputs from similar spaces.

Only need to instantiate the layer once and call it on as many inputs are required. __Shared layers have the same weights__

Below example applies the same embedding layer to different sequences of text with similar vocabulary

In [ ]:
# variable-length sequence of integers
text_input_a = tf.keras.Input(shape=(None, ), dtype='int32')
text_input_b = tf.keras.Input(shape=(None, ), dtype='int32')

# Create shared embedding layer
shared_embedding = tf.keras.layers.Embedding(1000, 128)

# Reuse the ame layer to encode both inputs 
encoded_input_a = shared_embedding(text_input_a)
encoded_input_b = shared_embedding(text_input_b)


## Extracting and reusing nodes in the graph of layers

Layer instances are objects that are chained to one another by the layer input and output tensors. The output of instantiated layers are tensors. 

The graph of layers is a static data structure. Keras allows easy access to inspect the model. Essentially this section explains different pre-trained models with weights, the start of transfer learning.

`tf.keras.application` module contains canned architectures with pre-trained weights

In [3]:
# pre-trained resnet model
resnet = tf.keras.applications.resnet.ResNet50()
intermediate_layers = [layer.output for layer in resnet.layers]
intermediate_layers[:10]

102973440/102967424 [==============================] - 8s 0us/step


[<tf.Tensor 'input_1:0' shape=(None, 224, 224, 3) dtype=float32>,
 <tf.Tensor 'conv1_pad/Identity:0' shape=(None, 230, 230, 3) dtype=float32>,
 <tf.Tensor 'conv1_conv/Identity:0' shape=(None, 112, 112, 64) dtype=float32>,
 <tf.Tensor 'conv1_bn/Identity:0' shape=(None, 112, 112, 64) dtype=float32>,
 <tf.Tensor 'conv1_relu/Identity:0' shape=(None, 112, 112, 64) dtype=float32>,
 <tf.Tensor 'pool1_pad/Identity:0' shape=(None, 114, 114, 64) dtype=float32>,
 <tf.Tensor 'pool1_pool/Identity:0' shape=(None, 56, 56, 64) dtype=float32>,
 <tf.Tensor 'conv2_block1_1_conv/Identity:0' shape=(None, 56, 56, 64) dtype=float32>,
 <tf.Tensor 'conv2_block1_1_bn/Identity:0' shape=(None, 56, 56, 64) dtype=float32>,
 <tf.Tensor 'conv2_block1_1_relu/Identity:0' shape=(None, 56, 56, 64) dtype=float32>]

In [4]:
# Select all feature layers - removes the prediction layer to get just the final weights for features
feature_layers = intermediate_layers[:-2]
# create feature extraction model
feat_extraction_model = keras.Model(inputs=resnet.input, outputs=feature_layers)

NameError: name 'interm' is not defined

## Transfer Learning
Deep learning models can be reused wholly or partly for other similar prediction problems. New model architecture is based on one or more layers from a pre-trained model. Weights fo the pre-trained models are used as the starting point and can be fixed,  fine-tuned or totally adapted during the learning phase.

2 main approaches:
1. Weights initialization
2. Feature extraction

# 3c. Keras Subclassing API

Keras is based on object-oriented programming, so able to subclass the `Model` class to create a new model architecture definition. It is also the third way to create deep neural network models.

Flexible, but more complex. Good for custom layers and unique more architectures. Also offers full control over the models and how to train them.

In [1]:
import tensorflow as tf
from tensorflow import keras

## Creating a customer layer

All layers are subclasses of the `Layer` class and implement the following methods

* `build` - defines the weights of the layer
* `call` - specifies the transformation from input to output by the layer
* `compute_output_shape` - perform automatic shape inference
* `get_config | from_config` - serialization and deserialization of layer

In [3]:
# create subclass layer from a custom dense layer
class MyCustomDense(tf.keras.layers.Layer):
    def __init__(self, units):
        super(MyCustomDense, self).__init__()
        self.units = units
    
    # define the weights and the bias
    def build(self, input_shape):
        self.w = self.add_weight(
			shape=(input_shape[-1], self.units),
			initializer='random_normal',
			trainable=True
		)

        self.b = self.add_weight(
			shape=(self.units, ),
			initializer='random_normal',
			trainable=True
		)

    # Applying this layer transformation to the input tensor
    def call(self, inputs):
        return tf.matmul(inputs, self.w) + self.b
	
	# function to retrieve the configuration
    def get_config(self):
	    return {'units': self.units}

In [4]:
x = tf.ones((2,2))
custom_layer = MyCustomDense(4)
y = custom_layer(x)
print(y) 

tf.Tensor(
[[-0.16667037 -0.0469715  -0.05102558 -0.10129508]
 [-0.16667037 -0.0469715  -0.05102558 -0.10129508]], shape=(2, 4), dtype=float32)


In [5]:
# Create model with the layer in the prev step
inputs = keras.Input((12, 4))
outputs = MyCustomDense(2)(inputs)
model = keras.Model(inputs, outputs)
config = model.get_config()

# Reloading model from config
new_model = keras.Model.from_config(
	config,
 	custom_objects={'MyCustomDense': MyCustomDense}
)

## Creating a custom model

Subclassign `tf.keras.Model` to build a fully customizable model

* Define layers in `__init__` method
* Control the forward pass of the model through the `call` method
* `training` boolean argument specify different behaviour during the training or inference phase

In [10]:
mnist = tf.keras.datasets.mnist
(X_mnist_train, y_mnist_train), (X_mnist_test, y_mnist_test) = mnist.load_data()
train_mnist_features = X_mnist_train/255
test_mnist_features = X_mnist_test/255

# Make subclass model for Mnist data]
class MyMNISTModel(tf.keras.Model):
    def __init__(self, num_classes):
        super(MyMNISTModel, self).__init__(name='my_mnist_model')
        self.num_classes = num_classes
        self.flatten = tf.keras.layers.Flatten()
        self.dropout = tf.keras.layers.Dropout(0.1)
        self.dense_1 = tf.keras.layers.Dense(50, activation='relu')
        self.dense_2 = tf.keras.layers.Dense(10, activation='softmax')
    
    def call(self, inputs, training=False):
        x = self.flatten(inputs)
        x = self.dense_1(x)
        if training:
            x = self.dropout(x)
        return self.dense_2(x)
	

In [11]:
# Instantiate the model and process the training
my_mnist_model = MyMNISTModel(10)
# compile
my_mnist_model.compile(
	optimizer='sgd',
	loss='sparse_categorical_crossentropy',
	metrics=['accuracy']
)
# train
my_mnist_model.fit(
	train_mnist_features,
	y_mnist_train,
	validation_data=(test_mnist_features, y_mnist_test),
	epochs=10
)

Epoch 1/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.7520 - accuracy: 0.7883 - val_loss: 0.3744 - val_accuracy: 0.9003
Epoch 2/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.3977 - accuracy: 0.8860 - val_loss: 0.3044 - val_accuracy: 0.9151
Epoch 3/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.3385 - accuracy: 0.9029 - val_loss: 0.2697 - val_accuracy: 0.9237
Epoch 4/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.3037 - accuracy: 0.9129 - val_loss: 0.2441 - val_accuracy: 0.9333
Epoch 5/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.2787 - accuracy: 0.9203 - val_loss: 0.2243 - val_accuracy: 0.9386
Epoch 6/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.2576 - accuracy: 0.9257 - val_loss: 0.2103 - val_accuracy: 0.9405
Epoch 7/10
1875/1875 [==============================] - 3s 2ms/step - loss: 0.2423 - accuracy: 0.9310 - val_loss: 0.1973 - val_accuracy:

Recommended to only use this method if unable to achieve using Sequential or Functional API. Complicated to implement but remains useful in a few cases, especially when need more control over different model aspects

# 3d. Keras Preprocessing API

Modules for data processing and data augmentation. API provides utilities for working with sequence, text and image data. Converts, transforms, or encodes raw data into an understandable and efficient format for algorithms,

Below are some of the key preprocessing methods from Keras for sequence, text and image data.

In [2]:
import tensorflow as tf
from tensorflow.keras import preprocessing
from tensorflow.keras.preprocessing.text import text_to_word_sequence, one_hot, hashing_trick, Tokenizer
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator, pad_sequences, skipgrams, make_sampling_table

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GRU

import numpy as np

import matplotlib.pyplot as plt

## Sequence Preprocessing

Sequence data is data where the order matters (e.g text, time series).

### Time series generator
Takes consecutive data points and applies transformations using time series parameters such as stride, length of history, etc. to create a TensorFlow dataset instance.

Keras generator helps to transform a univariate or multivariate time series dataset into a data structure ready to train models. Example of options to prepare data:
* shuffle
* sampling rate
* start and end offsets

In [15]:
# Toy time series data
series = np.array([i for i in range(10)])
print(series)

# Predict the next value from the last 5 lag observations
generator = TimeseriesGenerator(
	data=series,
	targets=series,
	length=5,
	batch_size=1,
	shuffle=False,
	reverse=False
)
print(f"Samples: {len(generator)}")

# Display the inputs and outputs of each sample to validate the training data
for i in range(len(generator)):
    x, y = generator[i]
    print(f"{x} => {y}")

[0 1 2 3 4 5 6 7 8 9]
Samples: 5
[[0 1 2 3 4]] => [5]
[[1 2 3 4 5]] => [6]
[[2 3 4 5 6]] => [7]
[[3 4 5 6 7]] => [8]
[[4 5 6 7 8]] => [9]


In [16]:
# Create and compile the model
model = Sequential()
model.add(Dense(10, activation='relu', input_dim=5))
model.add(Dense(1))
model.compile(optimizer='adam', loss='mse')

# Train the model
model.fit(generator, epochs=10)

Epoch 1/10
5/5 [==============================] - 0s 2ms/step - loss: 54.1877
Epoch 2/10
5/5 [==============================] - 0s 2ms/step - loss: 51.7320
Epoch 3/10
5/5 [==============================] - 0s 2ms/step - loss: 48.8416
Epoch 4/10
5/5 [==============================] - 0s 2ms/step - loss: 45.9309
Epoch 5/10
5/5 [==============================] - 0s 2ms/step - loss: 43.4346
Epoch 6/10
5/5 [==============================] - 0s 2ms/step - loss: 40.8733
Epoch 7/10
5/5 [==============================] - 0s 2ms/step - loss: 38.9168
Epoch 8/10
5/5 [==============================] - 0s 2ms/step - loss: 36.6037
Epoch 9/10
5/5 [==============================] - 0s 2ms/step - loss: 34.0424
Epoch 10/10
5/5 [==============================] - 0s 2ms/step - loss: 31.9781


### Padding sequences

Sequence data might have varying lengths, so padding helps to fit shorter ones into the desired length, while longer ones are truncated. Padding adds values to the end or beginning of each sequence

In [39]:
sentences = [["What", "do", "you", "like", "?"],
             ["I", "like", "basket-ball", "!"],
             ["And", "you", "?"],
             ["I", "like", "coconut", "and", "apple"]]

# Build vocabulary lookup table (int2vocab, vocab2int)
text_set = set(np.concatenate(sentences))
vocab_to_int = dict(zip(text_set, range(len(text_set))))
int_to_vocab = {vocab_to_int[word]: word for word in vocab_to_int}

encoded_sentences = []
for sentence in sentences:
    encoded_sentence = [vocab_to_int[word] for word in sentence]
    encoded_sentences.append(encoded_sentence)
print("Encoded Sentences ---")
print(encoded_sentences)

# Truncate and pad sequences to a common length (takes the longest sequence)
pad_sequences(encoded_sentences)

# Include post-sequence padding and set the maxlen argument to 7 (length)
longer = pad_sequences(encoded_sentences, maxlen=7)
print("Longer Sequence ---")
print(longer)

# Can also be Trimmed to desired length - removes from beginning 
shorter = pad_sequences(encoded_sentences, maxlen=3)
print("Shorter Sequence ---")
print(shorter)

# Setting truncating argument to post removes from the end of each sequence
front_rem = pad_sequences(encoded_sentences, maxlen=3, truncating='post')
print("Remove from front ---")
print(front_rem)

Encoded Sentences ---
[[0, 5, 1, 9, 4], [11, 9, 3, 10], [6, 1, 4], [11, 9, 2, 8, 7]]
Longer Sequence ---
[[ 0  0  0  5  1  9  4]
 [ 0  0  0 11  9  3 10]
 [ 0  0  0  0  6  1  4]
 [ 0  0 11  9  2  8  7]]
Shorter Sequence ---
[[ 1  9  4]
 [ 9  3 10]
 [ 6  1  4]
 [ 2  8  7]]
Remove from front ---
[[ 0  5  1]
 [11  9  3]
 [ 6  1  4]
 [11  9  2]]


Padding is useful when we want all sequences in a list to have the same length

### Skip-grams

_Unsupervised learning_ techniques that finds the most related words fro a given word and predicts the contect word for the given word.

It will take a pair of words and compare their relevance. If positive, it is set to 1, else label set to 0

In [44]:
# Encode a sentence as list of words
sentence = "I like coconut and apple !"
encoded_sentence = [vocab_to_int[word] for word in sentence.split()]
vocabulary_size = len(encoded_sentence)
print(f"Encoded Sentence: {encoded_sentence}")

# Create skipgram with window size 1
pairs, labels = skipgrams(
	encoded_sentence,
	vocabulary_size,
	window_size=1,
	negative_samples=0
)
for i in range(len(pairs)):
    print(f"({int_to_vocab[pairs[i][0]]} , {int_to_vocab[pairs[i][1]]}) -> ({labels[i]})")

Encoded Sentence: [11, 9, 2, 8, 7, 10]
(like , I) -> (1)
(I , like) -> (1)
(! , apple) -> (1)
(and , coconut) -> (1)
(coconut , and) -> (1)
(like , coconut) -> (1)
(apple , and) -> (1)
(and , apple) -> (1)
(apple , !) -> (1)
(coconut , like) -> (1)


## Text Preprocessing

Cannot feed raw text directly into the network. Encode text as numbers and provide integers as input

### Split text to word sequence

`text_to_word_sequece` transforms a sequence into a list of words or tokens.

In [45]:
sentence = "I like coconut , I like apple"
text_to_word_sequence(sentence, lower=False) # default splits on whitespace

['I', 'like', 'coconut', 'I', 'like', 'apple']

In [47]:
text_to_word_sequence(sentence, lower=True, filters=[]) # filter, filters out lists of characters like punctuations # filter, filters out lists of characters like punctuations # filter, filters out lists of characters like punctuations # filter, filters out lists of characters like punctuations

['i', 'like', 'coconut', ',', 'i', 'like', 'apple']

### Tokenizer

Utility class for text tokenization. Vectorize a text corpus, by turning each text into either a sequence of integers or into a vector where the coefficient of each token could be different data type (e.g binary, word count, etc.)

In [59]:
sentences = [["What", "do", "you", "like", "?"],
             ["I", "like", "basket-ball", "!"],
             ["And", "you", "?"],
             ["I", "like", "coconut", "and", "apple"]]

# Create tokenizer instance
t = Tokenizer()
t.fit_on_texts(sentences)

print(t.word_counts) # count of each word
print(t.document_count) # number of unique documents used to fit
print(t.word_index) # for each word, a unique integer identifier
print(t.word_docs) # for each word, the number of documents it appeared in

# Encode documents - text_to_matrix
binary = t.texts_to_matrix(sentences, mode='binary')
print("Binary ---")
print(binary)

count = t.texts_to_matrix(sentences, mode='count')
print("Count ---")
print(count)

tfidf = t.texts_to_matrix(sentences, mode='tfidf')
print("TF-IDF ---")
print(tfidf)

freq = t.texts_to_matrix(sentences, mode='freq')
print("Frequency ---")
print(freq)

OrderedDict([('what', 1), ('do', 1), ('you', 2), ('like', 3), ('?', 2), ('i', 2), ('basket-ball', 1), ('!', 1), ('and', 2), ('coconut', 1), ('apple', 1)])
4
{'like': 1, 'you': 2, '?': 3, 'i': 4, 'and': 5, 'what': 6, 'do': 7, 'basket-ball': 8, '!': 9, 'coconut': 10, 'apple': 11}
defaultdict(<class 'int'>, {'you': 2, '?': 2, 'what': 1, 'do': 1, 'like': 3, 'basket-ball': 1, 'i': 2, '!': 1, 'and': 2, 'coconut': 1, 'apple': 1})
Binary ---
[[0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 1. 0. 0. 0. 0. 1. 1.]]
Count ---
[[0. 1. 1. 1. 0. 0. 1. 1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 0. 0. 0. 1. 1. 0. 0.]
 [0. 0. 1. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 1. 0. 0. 1. 1. 0. 0. 0. 0. 1. 1.]]
TF-IDF ---
[[0.         0.69314718 0.84729786 0.84729786 0.         0.
  1.09861229 1.09861229 0.         0.         0.         0.        ]
 [0.         0.69314718 0.         0.         0.84729786 0.
  0.         0.         1.09861229 1.0

## Image Preprocessing

Real-time data augmentation on image data. `ImageDataGenerator` allows the creation of new data from the training dataset. Images created are not applied to the validation or testing set, only to increase the number of plausible examples in the training dataset. This is called data augmentation and it's not the same as data normalization or image resizing.

Examples: Field of image manipulation, rotations, brightness, etc.

Take note of the kind of augmentation you do, must make sense for the context of the image. E.g for MNIST doesn't make sense to flip the image since we're predicting handwritten digits

In [3]:
# Load CIFAR dataset
(x_cifar10_train, y_cifar10_train), (x_cifar10_test, y_cifar10_test) = tf.keras.datasets.cifar10.load_data()

# Preprocess the data - Normalization
x_cifar10_train = x_cifar10_train[:30000]
x_cifar10_test = x_cifar10_test
y_cifar10_train = y_cifar10_train[:30000]

# Image data generator that applies a horizontal flip, random rotation between 0-15 and shift 3 pixels on width and height
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
	rotation_range=15,
	width_shift_range=3,
	height_shift_range=3,
	horizontal_flip=True
)

# Create an iterator on the training set
it = datagen.flow(x_cifar10_train, y_cifar10_train, batch_size=10)

# Check if the images make sense 
plt.figure(figsize=(10, 10))
n_cols = 4
for i in range(n_cols):
    plt.subplot(1, n_cols, i + 1)
    batch = it.next()
    image = tf.squeeze(batch[0])
    plt.imshow(np.uint8(image))
plt.tight_layout()
plt.show()

   581632/170498071 [..............................] - ETA: 1:16:22

KeyboardInterrupt: 